# Parse House Roll Call PDF files

This notebook parses and saves data to CSV files from the House Roll Call vote PDFs that are stored in ```../../data/HJRollCalls'``` subfolders.

## Parsing a Single PDF File

First, I will parse the first House Roll Call Vote pdf in our dataset (stored at ```../../data/HJRollCalls/2003/hj01082003.pdf```)

This approach uses PyMuPDF to extract text from the PDF by line, then applies regular expressions to appropriately capture the data. This method works well because we know that the structure of the House Roll Call Vote Journals is consistent across the years, so regular expressions can capture the names of the different representatives as long as they are robust enough to handle some varied formatting.

In [56]:
import pymupdf
import re
import pandas as pd
from pathlib import Path

In [25]:
# Define regular expressions for validating a file's content
patterns_to_match =[
    '^\s+MASSACHUSETTS HOUSE OF REPRESENTATIVES\s+$',
    '^\s+Yea and Nay\s+(?P<RCDate>\d{2}/\d{2}/\d{2})\s(?P<RCTime>\d{2}:\d{2})\s\D',
    '^\s+No.\s+(?P<RCNumberForYear>\d+)\s+(?P<YeaVoteCount>\d+)\s+YEAS\s+(?P<NayVoteCount>\d+)\s+NAYS\s+(?P<NotVotingCount>\d+)\s+N/V\s+$',
    '^\s+(?P<Vote1>[Y,N,X,P])\s+(?P<Name1>Mr. Speaker|(.+?)(?:\s{,1}[A-Za-z.,\'-]+)+)\s+(?P<Vote2>[Y,N,X,P])\s+(?P<Name2>(.+?)(?:\s{,1}[A-Za-z.,\'-]+)+)\s+(?P<Vote3>[Y,N,X,P])\s+(?P<Name3>(.+?)(?:\s{,1}[A-Za-z.,\'-]+)+)\s+((?P<Vote4>[Y,N,X,P])\s+(?P<Name4>(.+?)(?:\s{,1}[A-Za-z.,\'-]+)+)\s+)?$',
    '^\s+\*=AFTER VOTE\s+P=PRESENT\s+X=NOT VOTING\s+$'
]

In [69]:
def parsePDF(path, bill_id, verbose=False):

    doc = pymupdf.open(path)
    all_pdf_bills = []
    all_pdf_votes = []

    for page in doc:
        text = page.get_text()
        lines = text.split('\n')

        current_pdf_bill_data = []
        current_pdf_bill_data.append(bill_id)

        current_pdf_vote_data = []

        for i, line in enumerate(lines):
            if verbose: print(f'{i}: {line}')

            # Skip parsing if blank line
            if line.isspace():
                if verbose: print("Found blank line")
                continue

            if i == 2: # parse full title line individually for ease
                current_pdf_bill_data.append(line.strip())

            # match = re.search(title_regex, line)
            matched = False
            for j, pattern in enumerate(patterns_to_match):
                match = re.match(pattern, line)

                if match:
                    matched = True
                    if verbose: print(f'Matched with pattern {pattern}')

                    if j == 0:
                        # Do nothing, just the title
                        if verbose: print("Parsing title line")
                    elif j == 1:
                        # Extract vote date and time
                        rc_date = match.group('RCDate')
                        current_pdf_bill_data.append(rc_date)
                        rc_time = match.group('RCTime')
                        current_pdf_bill_data.append(rc_time)
                        if verbose: print(f"Found date: {rc_date} and time: {rc_time}")
                    elif j == 2:
                        rc_count_for_year = match.group('RCNumberForYear')
                        current_pdf_bill_data.append(rc_count_for_year)

                        rc_yea_vote_count = match.group('YeaVoteCount')
                        current_pdf_bill_data.append(rc_yea_vote_count)

                        rc_nay_vote_count = match.group('NayVoteCount')
                        current_pdf_bill_data.append(rc_nay_vote_count)

                        rc_not_voting_count = match.group('NotVotingCount')
                        current_pdf_bill_data.append(rc_not_voting_count)

                        if verbose: print(f'Found {rc_yea_vote_count} YEA votes, {rc_nay_vote_count} Nay votes, {rc_not_voting_count} NV votes for RC Vote {rc_count_for_year}')
                    elif j == 3:
                        vote1 = match.group('Vote1')
                        name1 = match.group('Name1')
                        current_pdf_vote_data.append([bill_id, name1, vote1])

                        vote2 = match.group('Vote2')
                        name2 = match.group('Name2')
                        current_pdf_vote_data.append([bill_id, name2, vote2])

                        vote3 = match.group('Vote3')
                        name3 = match.group('Name3')
                        current_pdf_vote_data.append([bill_id, name3, vote3])

                        vote4 = match.group('Vote4')
                        name4 = match.group('Name4')
                        current_pdf_vote_data.append([bill_id, name4, vote4])

                        if verbose: print(f'Found {name1} with vote {vote1}, {name2} with vote {vote2}, {name3} with {vote3}')
                    elif j == 4:
                        if verbose: print("Parsing end line")

            if not matched:
                Exception(f"Failed to match regex on line {i} : {line}")

        all_pdf_bills.append(current_pdf_bill_data)
        all_pdf_votes.extend(current_pdf_vote_data)
        bill_id += 1

    new_bill_id = bill_id
    print(f"Succesfully parsed file at path: {path}")
    if verbose:
        print(f'Found {len(all_pdf_votes)} voting records')
        print(f'Found {len(all_pdf_bills)} bill records')
    return all_pdf_bills, all_pdf_votes, new_bill_id

In [70]:
bill_data = []
vote_data = []
bill_id = 0
path = "../../data/HJRollCalls/2003/hj01082003.pdf"

# Call defined function to get data for current pdf
pdf_bill_data, pdf_vote_data, bill_id = parsePDF(path, bill_id)

bill_data.extend(pdf_bill_data)
vote_data.extend(pdf_vote_data)

Succesfully parsed file at path: ../../data/HJRollCalls/2003/hj01082003.pdf


In [71]:
vote_data

[[0, 'Mr. Speaker', 'Y'],
 [0, 'Fresolo', 'Y'],
 [0, 'Murphy, K.J.', 'Y'],
 [0, 'Swan', 'Y'],
 [0, 'Asselin', 'Y'],
 [0, 'Galvin', 'Y'],
 [0, 'Nangle', 'Y'],
 [0, 'Teahan', 'Y'],
 [0, 'Atkins', 'Y'],
 [0, 'Garry', 'Y'],
 [0, 'Naughton', 'Y'],
 [0, 'Timilty', 'Y'],
 [0, 'Atsalis', 'Y'],
 [0, 'Gobi', 'X'],
 [0, 'Nyman', 'Y'],
 [0, 'Tobin', 'Y'],
 [0, 'Ayers', 'Y'],
 [0, 'Goguen', 'Y'],
 [0, "O'Brien", 'Y'],
 [0, 'Toomey', 'Y'],
 [0, 'Balser', 'Y'],
 [0, 'Golden B.P.', 'Y'],
 [0, "O'Flaherty", 'Y'],
 [0, 'Torrisi', 'Y'],
 [0, 'Binienda', 'Y'],
 [0, 'Golden T.A.', 'Y'],
 [0, 'Owens-Hicks', 'Y'],
 [0, 'Travis', 'Y'],
 [0, 'Blumer', 'Y'],
 [0, 'Grant', 'Y'],
 [0, 'Parente', 'Y'],
 [0, 'Turkington', 'Y'],
 [0, 'Bosley', 'Y'],
 [0, 'Greene', 'Y'],
 [0, 'Patrick', 'Y'],
 [0, 'Vallee', 'X'],
 [0, 'Bradley', 'Y'],
 [0, 'Haddad', 'Y'],
 [0, 'Paulsen', 'Y'],
 [0, 'Verga', 'X'],
 [0, 'Broadhurst', 'Y'],
 [0, 'Hall', 'Y'],
 [0, 'Pedone', 'Y'],
 [0, 'Wagner', 'Y'],
 [0, 'Buoniconti', 'Y'],
 [0, 'Harki

In [72]:
bill_data

[[0, 'QUORUM', '01/08/03', '01:44', '1', '151', '0', '9'],
 [1,
  'H2003  On motion to       Postpone',
  '01/08/03',
  '02:01',
  '2',
  '45',
  '107',
  '8'],
 [2, 'H2003  On Amendment', '01/08/03', '02:20', '3', '41', '110', '9'],
 [3, 'H2003  On Amendment', '01/08/03', '02:45', '4', '44', '107', '9'],
 [4, 'H2003  On Amendment', '01/08/03', '04:03', '5', '28', '119', '13'],
 [5, 'H2003  On Amendment', '01/08/03', '04:34', '6', '22', '128', '10']]

In [73]:
# Create a dataframe from the parsed PDF data for bills
bill_df = pd.DataFrame(bill_data, columns=['bill_id', 'bill_name', 'rollcall_date', 'rollcall_time', 'rollcall_count_for_year', 'yea_vote_count', 'nay_vote_count', 'not_voting_count'])

bill_df.set_index('bill_id', inplace=True)
bill_df

,bill_name,rollcall_date,rollcall_time,rollcall_count_for_year,yea_vote_count,nay_vote_count,not_voting_count
bill_id,,,,,,,
0,QUORUM,01/08/03,01:44,1,151,0,9
1,H2003 On motion to Postpone,01/08/03,02:01,2,45,107,8
2,H2003 On Amendment,01/08/03,02:20,3,41,110,9
3,H2003 On Amendment,01/08/03,02:45,4,44,107,9
4,H2003 On Amendment,01/08/03,04:03,5,28,119,13
5,H2003 On Amendment,01/08/03,04:34,6,22,128,10


In [74]:
# Create a dataframe from parsed PDF data about votes
vote_df = pd.DataFrame(vote_data, columns=['bill_id', 'voter_name', 'vote_value'])
vote_df

,bill_id,voter_name,vote_value
0,0,Mr. Speaker,Y
1,0,Fresolo,Y
2,0,"Murphy, K.J.",Y
3,0,Swan,Y
4,0,Asselin,Y
...,...,...,...
955,5,Webster,Y
956,5,Fox,N
957,5,"Murphy, J.M.",N
958,5,"Sullivan, J.C.",N


## Batch Processing all PDFs

Now that we have a function for scraping the data from all pages of a single pdf, we can loop through all of the pdfs that are stored on disk in the data and create the full dataset

In [75]:
# Get path to all pdfs stored in HJRollCall folder
all_paths = []
root_path = Path('../../data/HJRollCalls')

for path in root_path.glob('**/*.pdf'):
    all_paths.append(path)

print(len(all_paths))

599


In [89]:
for path in all_paths:
    file_year = int(str(path)[23:27])

    if file_year >= 2009 and file_year <= 2017:
        print(f'Skipping file at path: {path}')

Skipping file at path: ../../data/HJRollCalls/2013/hj04032013.pdf
Skipping file at path: ../../data/HJRollCalls/2013/hj07172013.pdf
Skipping file at path: ../../data/HJRollCalls/2013/hj09252013.pdf
Skipping file at path: ../../data/HJRollCalls/2013/hj11182013.pdf
Skipping file at path: ../../data/HJRollCalls/2013/hj10022013.pdf
Skipping file at path: ../../data/HJRollCalls/2013/hj06052013.pdf
Skipping file at path: ../../data/HJRollCalls/2013/hj07302013.pdf
Skipping file at path: ../../data/HJRollCalls/2013/hj01232013.pdf
Skipping file at path: ../../data/HJRollCalls/2013/hj11132013.pdf
Skipping file at path: ../../data/HJRollCalls/2013/hj10092013.pdf
Skipping file at path: ../../data/HJRollCalls/2013/hj04082013.pdf
Skipping file at path: ../../data/HJRollCalls/2013/hj04222013.pdf
Skipping file at path: ../../data/HJRollCalls/2013/hj06122013.pdf
Skipping file at path: ../../data/HJRollCalls/2013/hj11062013.pdf
Skipping file at path: ../../data/HJRollCalls/2013/hj07012013.pdf
Skipping f

In [96]:
bill_id = -1
bill_data = []
vote_data = []

for path in all_paths:

    # If path includes the year <2009, skip for now
    file_year = int(str(path)[23:27])

    if file_year >= 2009:
        print(f'Skipping file at path: {path}')
        continue

    print(f'Parsing file at path: {path}')
    current_bill_data, current_vote_data, bill_id = parsePDF(path, bill_id+1)

    bill_data.extend(current_bill_data)
    vote_data.extend(current_vote_data)

Skipping file at path: ../../data/HJRollCalls/2013/hj04032013.pdf
Skipping file at path: ../../data/HJRollCalls/2013/hj07172013.pdf
Skipping file at path: ../../data/HJRollCalls/2013/hj09252013.pdf
Skipping file at path: ../../data/HJRollCalls/2013/hj11182013.pdf
Skipping file at path: ../../data/HJRollCalls/2013/hj10022013.pdf
Skipping file at path: ../../data/HJRollCalls/2013/hj06052013.pdf
Skipping file at path: ../../data/HJRollCalls/2013/hj07302013.pdf
Skipping file at path: ../../data/HJRollCalls/2013/hj01232013.pdf
Skipping file at path: ../../data/HJRollCalls/2013/hj11132013.pdf
Skipping file at path: ../../data/HJRollCalls/2013/hj10092013.pdf
Skipping file at path: ../../data/HJRollCalls/2013/hj04082013.pdf
Skipping file at path: ../../data/HJRollCalls/2013/hj04222013.pdf
Skipping file at path: ../../data/HJRollCalls/2013/hj06122013.pdf
Skipping file at path: ../../data/HJRollCalls/2013/hj11062013.pdf
Skipping file at path: ../../data/HJRollCalls/2013/hj07012013.pdf
Skipping f

In [97]:
bill_data[0]

[0, 'S. 1188  land taking', '07/24/08', '02:14', '449', '153', '0', '5']

In [98]:
# Create a dataframe from the parsed PDF data for bills
bill_df = pd.DataFrame(bill_data, columns=['bill_id', 'bill_name', 'rollcall_date', 'rollcall_time', 'rollcall_count_for_year', 'yea_vote_count', 'nay_vote_count', 'not_voting_count'])

bill_df.set_index('bill_id', inplace=True)
bill_df

,bill_name,rollcall_date,rollcall_time,rollcall_count_for_year,yea_vote_count,nay_vote_count,not_voting_count
bill_id,,,,,,,
0,S. 1188 land taking,07/24/08,02:14,449,153,0,5
1,H. 4784 On Amendment,07/24/08,04:15,450,85,69,4
2,H. 4784 On Amendment,07/24/08,04:56,451,52,101,5
3,H. 4784 On Amendment,07/24/08,05:34,452,22,132,4
4,H. 4864 On Enactment,07/24/08,05:42,453,154,0,4
...,...,...,...,...,...,...,...
2077,H. 3899 On Enactment Land Taking,09/30/03,02:14,416,147,0,13
2078,QUORUM,09/30/03,02:55,417,150,0,10
2079,S. 187 On Engrossment,09/30/03,03:10,418,146,4,10


In [99]:
# Create a dataframe from parsed PDF data about votes
vote_df = pd.DataFrame(vote_data, columns=['bill_id', 'voter_name', 'vote_value'])
vote_df

,bill_id,voter_name,vote_value
0,0,Mr. Speaker,Y
1,0,Fallon,Y
2,0,Miceli,Y
3,0,Straus,Y
4,0,Rogers,Y
...,...,...,...
307551,2081,Webster,N
307552,2081,Festa,Y
307553,2081,Miceli,Y
307554,2081,"Sullivan, D.B.",Y


The parsed output can be written to csv files for future cleaning and processing

In [104]:
bill_df.to_csv('../../data/parsedcsv/hjbills_raw.csv', index=True, index_label=bill_df.index.name)
vote_df.to_csv('../../data/parsedcsv/hjrollcallvotes_raw.csv', index=False)

Read the files from CSV to test that they wrote properly

In [105]:
bill_df_from_csv = pd.read_csv('../../data/parsedcsv/hjbills_raw.csv')

bill_df_from_csv.head()

,bill_id,bill_name,rollcall_date,rollcall_time,rollcall_count_for_year,yea_vote_count,nay_vote_count,not_voting_count
0,0,S. 1188 land taking,07/24/08,02:14,449,153,0,5.0
1,1,H. 4784 On Amendment,07/24/08,04:15,450,85,69,4.0
2,2,H. 4784 On Amendment,07/24/08,04:56,451,52,101,5.0
3,3,H. 4784 On Amendment,07/24/08,05:34,452,22,132,4.0
4,4,H. 4864 On Enactment,07/24/08,05:42,453,154,0,4.0


In [107]:
vote_df_from_csv = pd.read_csv('../../data/parsedcsv/hjrollcallvotes_raw.csv')
vote_df_from_csv.head()

,bill_id,voter_name,vote_value
0,0,Mr. Speaker,Y
1,0,Fallon,Y
2,0,Miceli,Y
3,0,Straus,Y
4,0,Rogers,Y


## Debugging individual PDFs

In [95]:
path = '../../data/HJRollCalls/2022/hj06082022.pdf'
doc = pymupdf.open(path)

for page in doc:
    print("NEW DOC STARTING \n\n")
    text = page.get_text()
    lines = text.split('\n')

    for line in lines:
        print(line)
    

NEW DOC STARTING 


06/08/2022 01:36 PM
MASSACHUSETTS HOUSE OF REPRESENTATIVES
155 YEAS
0 NAYS
0 N/V
No. 194
Yea and Nay
 QUORUM
Mr. Speaker
Y
Farley-Bouvier
Y
Mahoney
Y
Turco
Y
Hogan
Y
Fernandes
Y
Malia
Y
Tyler
Y
Moran M.
Y
Ferrante
Y
Mark
Y
Ultrino
Y
Wagner
Y
Finn
Y
Markey
Y
Uyterhoeven
Y
Peake
Y
Fiola
Y
McGonagle
Y
Vargas
Y
Arciero
Y
Fluker Oakley
Y
McMurtry
Y
Vitolo
Y
Ashe
Y
Galvin
Y
Meschino
Y
Walsh
Y
Ayers
Y
Garballey
Y
Michlewitz
Y
Williams
Y
Balser
Y
Garlick
Y
Minicucci
Y
Zlotnik
Y
Barber
Y
Garry
Y
Miranda
Y
--Jones--
Y
Barrett
Y
Gentile
Y
Mom
Y
Ferguson
Y
Belsito
Y
Giannino
Y
Moran F.
Y
Frost
Y
Biele
Y
Gonzalez
Y
Murphy
Y
Gifford
Y
Blais
Y
Gordon
Y
Murray
Y
Barrows
Y
Cabral
Y
Gouveia
Y
Nguyen
Y
Berthiaume
Y
Cahill
Y
Gregoire
Y
O’Day
Y
Boldyga
Y
Campbell
Y
Haddad
Y
Oliveira
Y
D'Emilia
Y
Capano
Y
Haggerty
Y
Owens
Y
DeCoste
Y
Carey
Y
Hawkins
Y
Parisella
Y
Dooley
Y
Cassidy
Y
Hendricks
Y
Peisch
Y
Durant
Y
Chan
Y
Higgins
Y
Philips
Y
Howitt
Y
Ciccolo
Y
Holmes
Y
Pignatelli
Y
Kane
Y
Co

In [94]:
parsePDF(path, 0, verbose=True)

0:                          MASSACHUSETTS HOUSE OF REPRESENTATIVES                       
Matched with pattern ^\s+MASSACHUSETTS HOUSE OF REPRESENTATIVES\s+$
Parsing title line
1:   
Found blank line
2:                          H. 4343  On Enactment       Land Taking                      
3:   
Found blank line
4:   Yea and Nay                                                         01/16/08 02:07 P
Matched with pattern ^\s+Yea and Nay\s+(?P<RCDate>\d{2}/\d{2}/\d{2})\s(?P<RCTime>\d{2}:\d{2})\s\D
Found date: 01/16/08 and time: 02:07
5:    No.  244                     157 YEAS   0 NAYS   0 N/V                             
Matched with pattern ^\s+No.\s+(?P<RCNumberForYear>\d+)\s+(?P<YeaVoteCount>\d+)\s+YEAS\s+(?P<NayVoteCount>\d+)\s+NAYS\s+(?P<NotVotingCount>\d+)\s+N/V\s+$
Found 157 YEA votes, 0 Nay votes, 0 NV votes for RC Vote 244
6:   
Found blank line
7:   
Found blank line
8:   
Found blank line
9:   Y    Mr. Speaker     Y    Fennell         Y    Moran           Y    Sullivan       

([[0,
   'H. 4343  On Enactment       Land Taking',
   '01/16/08',
   '02:07',
   '244',
   '157',
   '0',
   '0'],
  [1,
   'H. 3741  On Enactment       Land Taking',
   '01/16/08',
   '02:15',
   '245',
   '156',
   '0',
   '1']],
 [[0, 'Mr. Speaker', 'Y'],
  [0, 'Fennell', 'Y'],
  [0, 'Moran', 'Y'],
  [0, 'Sullivan', 'Y'],
  [0, 'Rogers', 'Y'],
  [0, 'Fernandes', 'Y'],
  [0, 'Murphy, C.A.', 'Y'],
  [0, 'Swan', 'Y'],
  [0, 'Harkins', 'Y'],
  [0, 'Finegold', 'Y'],
  [0, 'Murphy, J.', 'Y'],
  [0, 'Timilty', 'Y'],
  [0, 'Rushing', 'Y'],
  [0, 'Flanagan', 'Y'],
  [0, 'Murphy, K.J.', 'Y'],
  [0, 'Tobin', 'Y'],
  [0, 'Petrolati', 'Y'],
  [0, 'Flynn', 'Y'],
  [0, 'Nangle', 'Y'],
  [0, 'Toomey', 'Y'],
  [0, 'Alicea', 'Y'],
  [0, 'Forry', 'Y'],
  [0, 'Natale', 'Y'],
  [0, 'Torrisi', 'Y'],
  [0, 'Allen', 'Y'],
  [0, 'Fox', 'Y'],
  [0, 'Naughton', 'Y'],
  [0, 'Turkington', 'Y'],
  [0, 'Atkins', 'Y'],
  [0, 'Fresolo', 'Y'],
  [0, 'Nyman', 'Y'],
  [0, 'Turner', 'Y'],
  [0, 'Atsalis', 'Y'],
  [0, 